In [1]:
# ============================================================
# INSTALL + GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets accelerate scikit-learn

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ============================================================
# IMPORT
# ============================================================

import os
import sys
import time
import random
import logging

from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd

import torch

from torch.utils.data import DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "distilbert-base-uncased"

DATASET_NAME = "glue"
DATASET_CFG  = "sst2"

NUM_LABELS = 2

TEXT_COLUMN  = "sentence"
LABEL_COLUMN = "label"

SETTING_TAG = "F-DistilBERT-FFT"

BATCH_SIZE = 32
LEARNING_RATE = 2e-5

ROUNDS = 20
LOCAL_EPOCHS = 1

MAX_LENGTH = 128

GRAD_ACCUM = 1

NUM_CLIENTS = 5

ALPHA = 0.5
PARTITION_TYPE = "dirichlet"

WARMUP_RATIO = 0.06
PATIENCE = 3

SEED = 42

OUTPUT_DIR = "/content/drive/MyDrive/fed_distilbert_sst2"

# ============================================================
# LOGGER
# ============================================================

def setup_logger(log_path: Path):

    log_path.parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(SETTING_TAG)

    logger.setLevel(logging.INFO)

    logger.handlers.clear()

    fmt = logging.Formatter(
        "[%(asctime)s] %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    fh = logging.FileHandler(
        log_path,
        mode="a",
        encoding="utf-8"
    )

    fh.setFormatter(fmt)

    sh = logging.StreamHandler(sys.stdout)

    sh.setFormatter(fmt)

    logger.addHandler(fh)

    logger.addHandler(sh)

    return logger

# ============================================================
# SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False

# ============================================================
# DATASET
# ============================================================

def load_and_tokenize(tokenizer, max_length):

    ds = load_dataset(DATASET_NAME, DATASET_CFG)

    train_ds = ds["train"]

    eval_ds = ds["validation"]

    def tok_fn(batch):

        return tokenizer(
            batch[TEXT_COLUMN],
            truncation=True,
            max_length=max_length
        )

    train_ds = train_ds.map(
        tok_fn,
        batched=True,
        remove_columns=[TEXT_COLUMN]
    )

    eval_ds = eval_ds.map(
        tok_fn,
        batched=True,
        remove_columns=[TEXT_COLUMN]
    )

    for c in list(train_ds.column_names):
        if c not in ("input_ids", "attention_mask", LABEL_COLUMN):
            train_ds = train_ds.remove_columns([c])

    for c in list(eval_ds.column_names):
        if c not in ("input_ids", "attention_mask", LABEL_COLUMN):
            eval_ds = eval_ds.remove_columns([c])

    train_ds = train_ds.rename_column(
        LABEL_COLUMN,
        "labels"
    )

    eval_ds = eval_ds.rename_column(
        LABEL_COLUMN,
        "labels"
    )

    train_ds.set_format("torch")

    eval_ds.set_format("torch")

    return train_ds, eval_ds

# ============================================================
# CLIENT PARTITION
# ============================================================

def partition_clients(
    labels,
    num_clients,
    partition_type,
    alpha,
    seed
):

    rng = np.random.default_rng(seed)

    n = len(labels)

    if partition_type == "iid":

        perm = rng.permutation(n)

        return [
            np.array(s)
            for s in np.array_split(perm, num_clients)
        ]

    labels = np.asarray(labels)

    num_classes = int(labels.max() + 1)

    client_idx = [[] for _ in range(num_clients)]

    for c in range(num_classes):

        idx_c = np.where(labels == c)[0]

        rng.shuffle(idx_c)

        prop = rng.dirichlet(
            alpha * np.ones(num_clients)
        )

        prop = (prop * len(idx_c)).astype(int)

        prop[-1] = (
            len(idx_c) - prop[:-1].sum()
        )

        start = 0

        for k, p in enumerate(prop):

            client_idx[k].extend(
                idx_c[start:start + p].tolist()
            )

            start += p

    return [
        np.array(idx)
        for idx in client_idx
    ]

# ============================================================
# MODEL
# ============================================================

def build_model():

    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS
    )

# ============================================================
# PARAM COUNT
# ============================================================

def count_params(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return trainable, total

# ============================================================
# COMMUNICATION COST
# ============================================================

def communication_cost_mb(model):

    return (
        sum(p.numel() for p in model.parameters())
        * 4
        / (1024 * 1024)
    )

# ============================================================
# LOCAL TRAIN
# ============================================================

def local_train(
    model,
    loader,
    device,
    scaler,
    num_steps_total
):

    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            WARMUP_RATIO * num_steps_total
        ),
        num_training_steps=num_steps_total,
    )

    losses = []

    t0 = time.time()

    n_samples = 0

    step = 0

    optimizer.zero_grad(set_to_none=True)

    for _ in range(LOCAL_EPOCHS):

        for batch in loader:

            batch = {
                k: v.to(device, non_blocking=True)
                for k, v in batch.items()
            }

            with autocast(dtype=torch.float16):

                outputs = model(**batch)

                loss = outputs.loss / GRAD_ACCUM

            scaler.scale(loss).backward()

            losses.append(loss.item() * GRAD_ACCUM)

            n_samples += batch["labels"].size(0)

            step += 1

            if step % GRAD_ACCUM == 0:

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                scaler.step(optimizer)

                scaler.update()

                scheduler.step()

                optimizer.zero_grad(set_to_none=True)

    elapsed = time.time() - t0

    state = {
        k: v.detach().cpu()
        for k, v in model.state_dict().items()
    }

    return (
        state,
        n_samples,
        float(np.mean(losses)),
        elapsed
    )

# ============================================================
# FEDAVG
# ============================================================

def fedavg(states, sizes):

    total = float(sum(sizes))

    weights = [
        c / total
        for c in sizes
    ]

    agg = OrderedDict()

    for key in states[0]:

        ref = states[0][key]

        if ref.is_floating_point():

            stacked = torch.stack([
                s[key].float() * w
                for s, w in zip(states, weights)
            ], dim=0)

            agg[key] = stacked.sum(0).to(ref.dtype)

        else:

            agg[key] = ref.clone()

    return agg

# ============================================================
# EVALUATE
# ============================================================

@torch.no_grad()
def evaluate(model, loader, device):

    model.eval()

    losses = []

    preds = []

    golds = []

    for batch in loader:

        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        with autocast(dtype=torch.float16):

            outputs = model(**batch)

        losses.append(outputs.loss.item())

        pred = outputs.logits.argmax(-1)

        preds.extend(pred.cpu().tolist())

        golds.extend(batch["labels"].cpu().tolist())

    metrics = {

        "eval_loss":
            float(np.mean(losses)),

        "accuracy":
            accuracy_score(golds, preds),

        "precision":
            precision_score(
                golds,
                preds,
                average="macro",
                zero_division=0
            ),

        "recall":
            recall_score(
                golds,
                preds,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                golds,
                preds,
                average="macro",
                zero_division=0
            ),
    }

    return metrics

# ============================================================
# CHECKPOINT
# ============================================================

class CheckpointManager:

    def __init__(self, directory, max_keep=2):

        self.directory = directory

        self.max_keep = max_keep

        directory.mkdir(parents=True, exist_ok=True)

    def save(self, payload, rnd):

        path = self.directory / f"checkpoint_round_{rnd:04d}.pt"

        torch.save(payload, path)

        self._prune()

        return path

    def _prune(self):

        ckpts = sorted(
            self.directory.glob("checkpoint_round_*.pt")
        )

        while len(ckpts) > self.max_keep:

            try:
                ckpts.pop(0).unlink()
            except OSError:
                pass

    def latest(self):

        ckpts = sorted(
            self.directory.glob("checkpoint_round_*.pt")
        )

        if len(ckpts) == 0:
            return None

        return ckpts[-1]

Mounted at /content/drive


In [2]:
# ============================================================
# MAIN
# ============================================================

def main():

    set_seed(SEED)

    out = Path(OUTPUT_DIR)

    out.mkdir(parents=True, exist_ok=True)

    ckpt_dir = out / "checkpoints"

    best_dir = out / "best_model"

    final_dir = out / "final_model"

    client_dir = out / "client_csv"

    client_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    logger = setup_logger(out / "train.log")

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    logger.info("=" * 70)
    logger.info(f"MODEL_NAME : {MODEL_NAME}")
    logger.info(f"SETTING    : {SETTING_TAG}")
    logger.info(f"DEVICE     : {device}")

    if torch.cuda.is_available():

        logger.info("RUNNING ON GPU")
        logger.info(f"GPU NAME        : {torch.cuda.get_device_name(0)}")

        total_vram = (
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3
        )

        logger.info(f"TOTAL VRAM      : {total_vram:.2f} GB")
        logger.info(f"CUDA VERSION    : {torch.version.cuda}")

    logger.info("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_ds, eval_ds = load_and_tokenize(
        tokenizer,
        MAX_LENGTH
    )

    collator = DataCollatorWithPadding(tokenizer)

    labels_arr = np.array(train_ds["labels"])

    client_idx = partition_clients(
        labels_arr,
        NUM_CLIENTS,
        PARTITION_TYPE,
        ALPHA,
        SEED
    )

    for k, idx in enumerate(client_idx):

        logger.info(f"Client {k}: {len(idx)} samples")

    client_loaders = [

        DataLoader(
            Subset(train_ds, list(idx)),
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=collator
        )

        for idx in client_idx
    ]

    eval_loader = DataLoader(
        eval_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        collate_fn=collator
    )

    global_model = build_model().to(device)

    trainable, total = count_params(global_model)

    comm_mb = communication_cost_mb(global_model)

    logger.info(
        f"trainable={trainable:,} total={total:,} "
        f"per-round MB={comm_mb:.2f}"
    )

    scaler = GradScaler()

    ckpt_mgr = CheckpointManager(ckpt_dir, max_keep=2)

    start_round = 1

    best_metric = -float("inf")

    patience_counter = 0

    latest = ckpt_mgr.latest()

    if latest is not None:

        logger.info(f"Resuming from {latest}")

        ckpt = torch.load(
            latest,
            map_location="cpu"
        )

        global_model.load_state_dict(
            ckpt["model"]
        )

        start_round = ckpt["round"] + 1

        best_metric = ckpt.get(
            "best_metric",
            -float("inf")
        )

        patience_counter = ckpt.get(
            "patience_counter",
            0
        )

    csv_path = out / "federated_training_results.csv"

    history = []

    # ========================================================
    # TRAINING LOOP
    # ========================================================

    for rnd in range(start_round, ROUNDS + 1):

        round_t0 = time.time()

        logger.info(
            f"==== Round {rnd}/{ROUNDS} ===="
        )

        global_state = {
            k: v.detach().cpu()
            for k, v in global_model.state_dict().items()
        }

        client_states = []

        sizes = []

        losses_ = []

        times_ = []

        for cid, loader in enumerate(client_loaders):

            local_model = build_model().to(device)

            local_model.load_state_dict(global_state)

            steps = max(
                1,
                len(loader) // GRAD_ACCUM
            )

            num_steps_total = (
                steps * LOCAL_EPOCHS
            )

            (
                state,
                n,
                tr_loss,
                ctime
            ) = local_train(
                local_model,
                loader,
                device,
                scaler,
                num_steps_total
            )

            logger.info(
                f"Client {cid}: "
                f"n={n} "
                f"loss={tr_loss:.4f} "
                f"time={ctime:.1f}s"
            )

            client_states.append(state)

            sizes.append(n)

            losses_.append(tr_loss)

            times_.append(ctime)

            del local_model

            torch.cuda.empty_cache()

        new_global = fedavg(
            client_states,
            sizes
        )

        global_model.load_state_dict(
            new_global
        )

        metrics = evaluate(
            global_model,
            eval_loader,
            device
        )

        round_time = time.time() - round_t0

        train_loss = float(
            np.average(
                losses_,
                weights=sizes
            )
        )

        avg_client_loss = float(
            np.mean(losses_)
        )

        is_new_best = (
            metrics["macro_f1"] > best_metric
        )

        if is_new_best:

            best_metric = metrics["macro_f1"]

            patience_counter = 0

            best_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            global_model.save_pretrained(best_dir)

            tokenizer.save_pretrained(best_dir)

        else:

            patience_counter += 1

        logger.info(
            f"acc={metrics['accuracy']:.4f} | "
            f"f1={metrics['macro_f1']:.4f} | "
            f"comm={comm_mb:.2f} MB"
        )

        ckpt_mgr.save({

            "round": rnd,

            "model": global_model.state_dict(),

            "best_metric": best_metric,

            "patience_counter": patience_counter,

        }, rnd)

        # ====================================================
        # MAIN CSV
        # ====================================================

        row = {

            "round": rnd,

            "train_loss": train_loss,

            "eval_loss": metrics["eval_loss"],

            "accuracy": metrics["accuracy"],

            "precision": metrics["precision"],

            "recall": metrics["recall"],

            "macro_f1": metrics["macro_f1"],

            "round_time": round_time,

            "trainable_params": trainable,

            "total_params": total,

            "communication_cost_MB": comm_mb,

            "communication_cost_MB_total_per_round":
                comm_mb * NUM_CLIENTS * 2,

            "client_avg_loss": avg_client_loss,

            "model_name": MODEL_NAME,

            "dataset_name": "glue/sst2",

            "setting": SETTING_TAG,

            "num_clients": NUM_CLIENTS,

            "local_epochs": LOCAL_EPOCHS,

            "partition_type": PARTITION_TYPE,

            "best_metric_so_far": best_metric,

            "patience_counter": patience_counter,

            "is_new_best": int(is_new_best),
        }

        for cid in range(NUM_CLIENTS):

            row[f"client_{cid}_loss"] = losses_[cid]

            row[f"client_{cid}_time"] = times_[cid]

            row[f"client_{cid}_samples"] = sizes[cid]

        history.append(row)

        df = pd.DataFrame(history)

        df.to_csv(
            csv_path,
            index=False
        )

        # ====================================================
        # CLIENT CSV
        # ====================================================

        for cid in range(NUM_CLIENTS):

            client_row = {

                "round": rnd,

                "client_id": cid,

                "client_loss": losses_[cid],

                "client_time": times_[cid],

                "num_samples": sizes[cid],

                "communication_cost_MB": comm_mb,

                "global_accuracy": metrics["accuracy"],

                "global_macro_f1": metrics["macro_f1"],

                "global_eval_loss": metrics["eval_loss"],
            }

            client_csv = (
                client_dir /
                f"client_{cid}.csv"
            )

            client_df = pd.DataFrame([client_row])

            if client_csv.exists():

                old_client_df = pd.read_csv(client_csv)

                client_df = pd.concat(
                    [old_client_df, client_df],
                    ignore_index=True
                )

            client_df.to_csv(
                client_csv,
                index=False
            )

        logger.info(
            f"CSV SAVED -> {csv_path}"
        )

        if patience_counter >= PATIENCE:

            logger.info(
                f"Early stopping at round {rnd}"
            )

            break

    final_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    global_model.save_pretrained(final_dir)

    tokenizer.save_pretrained(final_dir)

    logger.info(
        f"Done. Best macro_f1={best_metric:.4f}"
    )

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()

[2026-05-18 01:44:29] INFO | ======================================================================


INFO:F-DistilBERT-FFT:======================================================================


[2026-05-18 01:44:29] INFO | MODEL_NAME : distilbert-base-uncased


INFO:F-DistilBERT-FFT:MODEL_NAME : distilbert-base-uncased


[2026-05-18 01:44:29] INFO | SETTING    : F-DistilBERT-FFT


...

[2026-05-18 02:05:04] INFO | Early stopping at round 9


INFO:F-DistilBERT-FFT:Early stopping at round 9


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-05-18 02:05:07] INFO | Done. Best macro_f1=0.9082


INFO:F-DistilBERT-FFT:Done. Best macro_f1=0.9082
